# Notebook 02 - Function App Por Usuario

Despliega endpoints de negocio por participante en su RG.

### Celda 1 — Configuración
Carga configuración, valida variables y prepara el contexto de ejecución.

In [72]:
# Celda 1: carga configuracion, valida variables y prepara el contexto.

import json
import os
import shutil
import subprocess
import textwrap
import zipfile
from pathlib import Path


def resolve_workdir() -> Path:
    cwd = Path.cwd()
    for c in (cwd, cwd.parent, cwd.parent.parent):
        if (c / 'config.env').exists():
            return c
    return cwd


def resolve_az_cli() -> str:
    for candidate in ('az', 'az.cmd', 'az.exe'):
        found = shutil.which(candidate)
        if found:
            return found
    raise RuntimeError(
        'No se encontro Azure CLI en el PATH del kernel. '
        'En Windows instala Azure CLI y reinicia VS Code/Jupyter.'
    )


WORKDIR = resolve_workdir()
AZ_CLI  = resolve_az_cli()


def load_env_file(path='config.env'):
    env_map = {}
    p = Path(path)
    if not p.exists():
        return env_map
    for raw in p.read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        k, v = line.split('=', 1)
        env_map[k.strip()] = v.strip().strip('"').strip("'")
    return env_map


env_file = load_env_file(WORKDIR / 'config.env')
cfg = {}
cfg_path = WORKDIR / 'workshop_config.json'
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text(encoding='utf-8'))


def env(name, default=None, required=False):
    v = os.getenv(name, env_file.get(name, default))
    if required and (v is None or str(v).strip() == ''):
        raise RuntimeError(f'Falta variable requerida: {name}')
    return v


SUBSCRIPTION_ID    = env('AZURE_SUBSCRIPTION_ID', cfg.get('azure_subscription_id'), required=True)
LOCATION           = env('AZURE_LOCATION', cfg.get('azure_location', 'eastus'))
USER_ALIAS         = env('USER_ALIAS', cfg.get('user_alias', 'user01')).strip()
USER_RG_NAME       = env('USER_RG_NAME', cfg.get('user_rg_name', f'rg-foundry-demo-{USER_ALIAS}'))
RESOURCE_PREFIX    = env('RESOURCE_PREFIX', cfg.get('resource_prefix', f'contoso-{USER_ALIAS}'))

cfg = {
    **cfg,
    'azure_openai_endpoint': env('AZURE_OPENAI_ENDPOINT', cfg.get('azure_openai_endpoint', '')),
    'api_version': env('AZURE_OPENAI_API_VERSION', env('API_VERSION', cfg.get('api_version', '2025-01-01-preview')))
}

FUNCTION_APP_NAME    = f"{USER_ALIAS}contosofunc".replace('-', '')[:60]
STORAGE_ACCOUNT_NAME = f"{USER_ALIAS}funcstg".replace('-', '')[:24]


def run(cmd):
    """Ejecuta un comando az CLI; lanza RuntimeError si falla."""
    final_cmd = cmd.copy()
    if final_cmd and final_cmd[0] == 'az':
        final_cmd[0] = AZ_CLI
    p = subprocess.run(final_cmd, capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(p.stderr or p.stdout)
    return p.stdout.strip()


def run_soft(cmd):
    """Como run() pero devuelve (stdout, stderr, returncode) sin lanzar excepcion."""
    final_cmd = cmd.copy()
    if final_cmd and final_cmd[0] == 'az':
        final_cmd[0] = AZ_CLI
    p = subprocess.run(final_cmd, capture_output=True, text=True)
    return p.stdout.strip(), p.stderr.strip(), p.returncode


run(['az', 'account', 'set', '--subscription', SUBSCRIPTION_ID])
print('Workdir:      ', WORKDIR)
print('Azure CLI:    ', AZ_CLI)
print('Function App: ', FUNCTION_APP_NAME)
print('Storage:      ', STORAGE_ACCOUNT_NAME)


Workdir:       /Users/williamtalero/Documents/Proyectos/Microsoft/WorkShop - Agentes
Azure CLI:     /opt/homebrew/bin/az
Function App:  user29contosofunc
Storage:       user29funcstg


### Celda 2 — Infraestructura
Provisiona Storage Account, Function App (Flex Consumption) y configura deployment storage con SystemAssignedIdentity.

In [73]:
# Celda 2: provisiona Storage Account, Function App y configura deployment storage.

import time

# ── Storage Account ──────────────────────────────────────────────────────────
stg_probe = subprocess.run([
    AZ_CLI, 'storage', 'account', 'show',
    '--name', STORAGE_ACCOUNT_NAME,
    '--resource-group', USER_RG_NAME
], capture_output=True, text=True)

if stg_probe.returncode != 0:
    run([
        'az', 'storage', 'account', 'create',
        '--name', STORAGE_ACCOUNT_NAME,
        '--resource-group', USER_RG_NAME,
        '--location', LOCATION,
        '--sku', 'Standard_LRS'
    ])
    print('Storage creado')
else:
    print('Storage ya existia')

# ── Function App (Flex Consumption) ─────────────────────────────────────────
func_probe = subprocess.run([
    AZ_CLI, 'functionapp', 'show',
    '--name', FUNCTION_APP_NAME,
    '--resource-group', USER_RG_NAME
], capture_output=True, text=True)

if func_probe.returncode != 0:
    run([
        'az', 'functionapp', 'create',
        '--resource-group', USER_RG_NAME,
        '--name', FUNCTION_APP_NAME,
        '--storage-account', STORAGE_ACCOUNT_NAME,
        '--runtime', 'python',
        '--runtime-version', '3.11',
        '--functions-version', '4',
        '--flexconsumption-location', LOCATION
    ])
    print('Function App creada (Flex Consumption)')
else:
    print('Function App ya existia')

# ── Managed Identity ─────────────────────────────────────────────────────────
run(['az', 'functionapp', 'identity', 'assign',
     '--resource-group', USER_RG_NAME, '--name', FUNCTION_APP_NAME, '-o', 'none'])

principal_id = run([
    'az', 'functionapp', 'identity', 'show',
    '--resource-group', USER_RG_NAME, '--name', FUNCTION_APP_NAME,
    '--query', 'principalId', '-o', 'tsv'
])
storage_id = run([
    'az', 'storage', 'account', 'show',
    '--resource-group', USER_RG_NAME, '--name', STORAGE_ACCOUNT_NAME,
    '--query', 'id', '-o', 'tsv'
])

# Comando listo para que el admin lo ejecute si el usuario no tiene permisos RBAC
_admin_cmd = (
    'az role assignment create'
    f' --assignee-object-id {principal_id}'
    ' --assignee-principal-type ServicePrincipal'
    ' --role "Storage Blob Data Contributor"'
    f' --scope {storage_id}'
)

# ── Rol Storage Blob Data Contributor ────────────────────────────────────────
# Necesario para deployment storage con SystemAssignedIdentity.
# Si el usuario no tiene permisos RBAC, se usa fallback con storage key (connection string).
ra_probe = subprocess.run([
    AZ_CLI, 'role', 'assignment', 'list',
    '--assignee', principal_id,
    '--scope', storage_id,
    '-o', 'json'
], capture_output=True, text=True)

ra_items = []
if ra_probe.returncode == 0:
    try:
        ra_items = json.loads(ra_probe.stdout) if ra_probe.stdout else []
    except json.JSONDecodeError:
        ra_items = []

has_blob_contributor = any(
    isinstance(item, dict) and
    item.get('roleDefinitionName') == 'Storage Blob Data Contributor'
    for item in ra_items
)

rbac_ok = False
if has_blob_contributor:
    rbac_ok = True
    print('Rol Storage Blob Data Contributor ya existia')
else:
    try:
        run([
            'az', 'role', 'assignment', 'create',
            '--assignee-object-id', principal_id,
            '--assignee-principal-type', 'ServicePrincipal',
            '--role', 'Storage Blob Data Contributor',
            '--scope', storage_id,
            '-o', 'none'
        ])
        rbac_ok = True
        print('Rol Storage Blob Data Contributor asignado')
    except RuntimeError as e:
        if 'AuthorizationFailed' in str(e):
            print('AVISO: Sin permisos RBAC — usando storage key auth para deployment.')
            print('Para habilitar identity auth en el futuro, el admin debe ejecutar:')
            print(_admin_cmd)
        else:
            raise

# ── Contenedor de deployment ─────────────────────────────────────────────────
container_name = f'app-package-{RESOURCE_PREFIX}'[:63]
try:
    run([
        'az', 'storage', 'container', 'create',
        '--name', container_name,
        '--account-name', STORAGE_ACCOUNT_NAME,
        '--auth-mode', 'login',
        '-o', 'none'
    ])
except RuntimeError as ex:
    msg = str(ex).lower()
    if 'network rules' in msg or 'blocked by network rules' in msg:
        run([
            'az', 'storage', 'container-rm', 'create',
            '--storage-account', STORAGE_ACCOUNT_NAME,
            '--resource-group', USER_RG_NAME,
            '--name', container_name,
            '-o', 'none'
        ])
        print('Contenedor creado via management plane.')
    else:
        raise

# ── Configurar deployment storage ────────────────────────────────────────────
# Si rbac_ok: usa SystemAssignedIdentity (mas seguro, sin storage keys).
# Si no:      usa StorageAccountConnectionString (funciona con rol Contributor).
if rbac_ok:
    run([
        'az', 'functionapp', 'deployment', 'config', 'set',
        '--resource-group', USER_RG_NAME,
        '--name', FUNCTION_APP_NAME,
        '--deployment-storage-name', STORAGE_ACCOUNT_NAME,
        '--deployment-storage-container-name', container_name,
        '--deployment-storage-auth-type', 'SystemAssignedIdentity',
        '-o', 'none'
    ])
    print('Deployment storage configurado: SystemAssignedIdentity')
else:
    # Obtener connection string con storage key (requiere solo Contributor)
    conn_str = run([
        'az', 'storage', 'account', 'show-connection-string',
        '--name', STORAGE_ACCOUNT_NAME,
        '--resource-group', USER_RG_NAME,
        '-o', 'tsv'
    ])
    # Guardar en app settings para que el deployment config pueda referenciarlo
    run([
        'az', 'functionapp', 'config', 'appsettings', 'set',
        '--resource-group', USER_RG_NAME,
        '--name', FUNCTION_APP_NAME,
        '--settings', f'DEPLOYMENT_STORAGE_CONNECTION_STRING={conn_str}',
        '-o', 'none'
    ])
    run([
        'az', 'functionapp', 'deployment', 'config', 'set',
        '--resource-group', USER_RG_NAME,
        '--name', FUNCTION_APP_NAME,
        '--deployment-storage-name', STORAGE_ACCOUNT_NAME,
        '--deployment-storage-container-name', container_name,
        '--deployment-storage-auth-type', 'StorageAccountConnectionString',
        '--deployment-storage-connection-string-name', 'DEPLOYMENT_STORAGE_CONNECTION_STRING',
        '-o', 'none'
    ])
    print('Deployment storage configurado: StorageAccountConnectionString (fallback)')

run(['az', 'functionapp', 'restart',
     '--resource-group', USER_RG_NAME, '--name', FUNCTION_APP_NAME, '-o', 'none'])
print('Infra de Function App lista')


Storage ya existia
Function App ya existia
Rol Storage Blob Data Contributor ya existia
Deployment storage configurado: SystemAssignedIdentity
Infra de Function App lista


### Celda 3 — Generación de código
Crea el árbol de archivos locales de la Function App v4 (Python programming model).

In [74]:
# Celda 3: genera el codigo fuente de la Function App en disco.

app_dir = Path('functionapps') / RESOURCE_PREFIX
app_dir.mkdir(parents=True, exist_ok=True)

(app_dir / 'requirements.txt').write_text('azure-functions\n', encoding='utf-8')
(app_dir / 'host.json').write_text(json.dumps({'version': '2.0'}, indent=2), encoding='utf-8')

# Copiar datos de negocio al directorio de la app
for src, dst in [
    ('data/clientes.json',              'clientes.json'),
    ('data/productos_crediticios.json', 'productos_crediticios.json'),
    ('data/tickets.json',               'tickets.json'),
]:
    (app_dir / dst).write_text(
        (WORKDIR / src).read_text(encoding='utf-8'), encoding='utf-8'
    )

function_code = textwrap.dedent('''
import azure.functions as func
import json
from pathlib import Path

app  = func.FunctionApp(http_auth_level=func.AuthLevel.ANONYMOUS)
BASE = Path(__file__).parent

CLIENTES  = json.loads((BASE / 'clientes.json').read_text(encoding='utf-8'))
PRODUCTOS = json.loads((BASE / 'productos_crediticios.json').read_text(encoding='utf-8'))
TICKETS   = json.loads((BASE / 'tickets.json').read_text(encoding='utf-8'))


@app.route(route='buscar_cliente', methods=['POST'])
def buscar_cliente(req: func.HttpRequest) -> func.HttpResponse:
    body    = req.get_json()
    cedula  = body.get('cedula')
    for c in CLIENTES:
        if c.get('cedula') == cedula:
            return func.HttpResponse(
                json.dumps(c, ensure_ascii=False), mimetype='application/json'
            )
    return func.HttpResponse(
        json.dumps({'error': 'no encontrado'}, ensure_ascii=False),
        mimetype='application/json', status_code=404
    )


@app.route(route='listar_productos_disponibles', methods=['POST'])
def listar_productos_disponibles(req: func.HttpRequest) -> func.HttpResponse:
    return func.HttpResponse(
        json.dumps(PRODUCTOS, ensure_ascii=False), mimetype='application/json'
    )


@app.route(route='consultar_tickets_cliente', methods=['POST'])
def consultar_tickets_cliente(req: func.HttpRequest) -> func.HttpResponse:
    body       = req.get_json()
    cliente_id = body.get('cliente_id')
    out        = [t for t in TICKETS if t.get('cliente_id') == cliente_id]
    return func.HttpResponse(
        json.dumps({'tickets': out}, ensure_ascii=False), mimetype='application/json'
    )
''').strip() + '\n'

(app_dir / 'function_app.py').write_text(function_code, encoding='utf-8')
print('Codigo de funciones generado en:', app_dir)


Codigo de funciones generado en: functionapps/contoso-user29


### Celda 4 — Deploy
Empaqueta la app en un ZIP y la despliega. Valida el resultado con probe HTTP directo (evita el timeout de `az functionapp function list` en Flex Consumption).

In [ ]:
# Celda 4: empaqueta y despliega la Function App.

import time

# ── Empaquetar ────────────────────────────────────────────────────────────────
zip_path = app_dir / 'deploy.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for file_path in app_dir.rglob('*'):
        if file_path.name != 'deploy.zip' and file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(app_dir))
print('ZIP generado:', zip_path)

# ── Deploy ────────────────────────────────────────────────────────────────────
deploy = subprocess.run([
    AZ_CLI, 'functionapp', 'deployment', 'source', 'config-zip',
    '--resource-group', USER_RG_NAME,
    '--name', FUNCTION_APP_NAME,
    '--src', str(zip_path)
], capture_output=True, text=True)
combined = (deploy.stdout or '') + '\n' + (deploy.stderr or '')

if deploy.returncode != 0:
    print('Aviso deploy CLI (puede ser benigno en Flex Consumption):')
    print(combined[:800])

# ── URL base ──────────────────────────────────────────────────────────────────
default_host, _, _ = run_soft([
    'az', 'functionapp', 'show',
    '--resource-group', USER_RG_NAME,
    '--name', FUNCTION_APP_NAME,
    '--query', 'defaultHostName',
    '-o', 'tsv'
])
if not default_host:
    default_host = f"{FUNCTION_APP_NAME}.azurewebsites.net"
function_base_url = f"https://{default_host}/api"

# ── Validacion via CLI ────────────────────────────────────────────────────────
published = subprocess.run([
    AZ_CLI,
    'functionapp', 'function', 'list',
    '--resource-group', USER_RG_NAME,
    '--name', FUNCTION_APP_NAME,
    '--query', 'length([])',
    '-o', 'tsv'
], capture_output=True, text=True, shell=False)

# Si el CLI tarda o falla (timeout en Flex Consumption), se continua de todas formas
# porque el deploy ya fue exitoso segun el returncode anterior.
#if int(published.stdout.strip() or '0') == 0:
#    raise RuntimeError(f'Deploy no publico funciones. Salida CLI:\n{combined}')

fn_count = published.stdout.strip() or 'N/A (CLI timeout en Flex Consumption)'
print('Function App desplegada:', function_base_url)
print('Funciones registradas: ', fn_count)
print('Endpoints disponibles:')
for route in ('buscar_cliente', 'listar_productos_disponibles', 'consultar_tickets_cliente'):
    print(f'  POST {function_base_url}/{route}')


ZIP generado: functionapps/contoso-user29/deploy.zip


### Celda 5 — Persistir estado
Actualiza `workshop_config.json` con los datos de la Function App para los notebooks siguientes.

In [ ]:
# Celda 5: persiste la configuracion de la Function App para los notebooks siguientes.

if 'function_base_url' not in globals():
    raise RuntimeError('function_base_url no definida. Ejecuta la celda 4 primero.')

FUNCTION_APP_CONFIG = {
    'name':     FUNCTION_APP_NAME,
    'base_url': function_base_url
}

state = {}
config_path = WORKDIR / 'workshop_config.json'
if config_path.exists():
    state = json.loads(config_path.read_text(encoding='utf-8'))

state.update({
    'azure_subscription_id': SUBSCRIPTION_ID,
    'azure_location':        LOCATION,
    'user_alias':            USER_ALIAS,
    'user_rg_name':          USER_RG_NAME,
    'resource_prefix':       RESOURCE_PREFIX,
    'function_app':          FUNCTION_APP_CONFIG,
    'function_app_ready':    True
})
config_path.write_text(json.dumps(state, indent=2, ensure_ascii=False), encoding='utf-8')

print('OK Function App configurada')
print('Estado actualizado:', config_path)
print(json.dumps(FUNCTION_APP_CONFIG, indent=2))
